<a href="https://colab.research.google.com/github/EberHernandezBenitez/EDP/blob/main/L%C3%ADnea_espera_en_serie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de línea de espera con dos servidores en serie.

Consideremos un sistema de dos servidores en el que los clientes llegan de acuerdo con un proceso de Poisson no homogéneo y supongamos que cada cliente debe ser atendida primero por el servidor 1 y al terminar el servicio en 1, el cliente pasa al servidor 2.

Lo que nos dice es que se podrán formas dos líneas de espera, la primera al esperar que pase el cliente al servidor 1 y la otra,esperar que el cliente pase al servidor 2 después de pasar al servidor 1.

Parametros: \
    - T: Tiempo máximo de simulación (límite para aceptar llegadas).\
    - tasa_arribo: Parámetro lambda para el proceso de Poisson de entrada.\
    - tasa_servicio1: Parámetro mu para el tiempo de servicio del servidor 1.\
    - tasa_servicio2: Parámetro mu para el tiempo de servicio del servidor 2.\
    - t Variable de tiempo\
    - N_A        Contador de llegadas al sistema\
    - N_D       Contador de salidas del sistema (servidor 2)\
    - n1       Clientes en el servidor 1 (en servicio + en cola)\
    - n2        Clientes en el servidor 2 (en servicio + en cola) \

    
  **A continuación realizamos el código para simular y dar el tiempo que pasa el cliente en ambos servidores.**

In [4]:
import random

def simular_servidores_en_serie(T, tasa_arribo, tasa_servicio1, tasa_servicio2):
    """
    Simulación de un sistema de línea de espera con dos servidores en serie.
    """
    # INICIALIZACIÓN
    t = 0        # Variable de tiempo
    N_A = 0      # Contador de llegadas al sistema
    N_D = 0      # Contador de salidas del sistema (servidor 2)
    n1 = 0       # Clientes en el servidor 1 (en servicio + en cola)
    n2 = 0       # Clientes en el servidor 2 (en servicio + en cola)

    t_A = random.expovariate(tasa_arribo)
    t1 = float('inf')
    t2 = float('inf')

    # Diccionarios para registrar los tiempos de cada cliente
    A1 = {}  # Hora de llegada al servidor 1
    A2 = {}  # Hora de llegada al servidor 2 (salida de 1)
    D = {}   # Hora de salida del servidor 2 (salida del sistema)

    # BUCLE PRINCIPAL DE SIMULACIÓN
    while t_A < float('inf') or t1 < float('inf') or t2 < float('inf'):
        proximo_evento = min(t_A, t1, t2)

        #CASO 1: Llegada al Servidor 1
        if proximo_evento == t_A:
            t = t_A
            N_A += 1
            n1 += 1

            T_r = random.expovariate(tasa_arribo)
            if t + T_r <= T:
                t_A = t + T_r
            else:
                t_A = float('inf')

            if n1 == 1:
                Y1 = random.expovariate(tasa_servicio1)
                t1 = t + Y1

            A1[N_A] = t

        #CASO 2: Fin de servicio en el Servidor 1 / Llegada al Servidor 2
        elif proximo_evento == t1:
            t = t1
            n1 -= 1
            n2 += 1

            if n1 == 0:
                t1 = float('inf')
            else:
                Y1 = random.expovariate(tasa_servicio1)
                t1 = t + Y1

            if n2 == 1:
                Y2 = random.expovariate(tasa_servicio2)
                t2 = t + Y2

            A2[N_A - n1] = t

        #CASO 3: Fin de servicio en el Servidor 2 / Salida del sistema
        elif proximo_evento == t2:
            t = t2
            N_D += 1
            n2 -= 1

            if n2 == 0:
                t2 = float('inf')
            else:
                Y2 = random.expovariate(tasa_servicio2)
                t2 = t + Y2

            D[N_D] = t

    return A1, A2, D
#Para calcular los tiempos promedios:
def calcular_estadisticas_tiempos(A1, A2, D):

    total_clientes_atendidos = len(D)

    if total_clientes_atendidos == 0:
        print("No hubo clientes que completaran el sistema.")
        return

    suma_tiempo_servidor1 = 0.0
    suma_tiempo_servidor2 = 0.0
    suma_tiempo_sistema = 0.0

    for i in range(1, total_clientes_atendidos + 1):
        # Tiempo en el servidor 1 = Llegada al servidor 2 - Llegada al servidor 1
        t_s1 = A2[i] - A1[i]
        # Tiempo en el servidor 2 = Salida del sistema - Llegada al servidor 2
        t_s2 = D[i] - A2[i]
        # Tiempo total en el sistema = Salida del sistema - Llegada al servidor 1
        t_sistema = D[i] - A1[i]

        suma_tiempo_servidor1 += t_s1
        suma_tiempo_servidor2 += t_s2
        suma_tiempo_sistema += t_sistema

    # Promedios
    promedio_s1 = suma_tiempo_servidor1 / total_clientes_atendidos
    promedio_s2 = suma_tiempo_servidor2 / total_clientes_atendidos
    promedio_sistema = suma_tiempo_sistema / total_clientes_atendidos

    return promedio_s1, promedio_s2, promedio_sistema

# EJEMPLO DE USO Y PRUEBA
if __name__ == "__main__":
    # Configuración de la simulación
    TIEMPO_MAX = 100       # Tiempo máximo para permitir llegadas (T)
    LAMBDA_ARR = 1.5       # Tasa de arribos (λ)
    MU_SERV1   = 2.5       # Tasa de servicio Servidor 1 (μ1)
    MU_SERV2   = 2.0       # Tasa de servicio Servidor 2 (μ2)

    # Ejecución
    arribos_s1, arribos_s2, salidas = simular_servidores_en_serie(
        TIEMPO_MAX, LAMBDA_ARR, MU_SERV1, MU_SERV2
    )

    print(f"Simulación completa. Total de clientes que salieron del sistema: {len(salidas)}")

    # Promedios
    prom_s1, prom_s2, prom_sis = calcular_estadisticas_tiempos(arribos_s1, arribos_s2, salidas)
    print("="*50)
    print(f"Tiempo promedio en el Servidor 1 (Cola + Servicio): {prom_s1:.4f} unidades de tiempo")
    print(f"Tiempo promedio en el Servidor 2 (Cola + Servicio): {prom_s2:.4f} unidades de tiempo")
    print(f"Tiempo promedio total en el Sistema:                {prom_sis:.4f} unidades de tiempo")
    print("="*50)

Simulación completa. Total de clientes que salieron del sistema: 156
Tiempo promedio en el Servidor 1 (Cola + Servicio): 0.7365 unidades de tiempo
Tiempo promedio en el Servidor 2 (Cola + Servicio): 2.7742 unidades de tiempo
Tiempo promedio total en el Sistema:                3.5107 unidades de tiempo


Con este código logramos calcular los tiempos promedios en le servidor 1, servidor 2 y el tiempo promedio en el sistema de un cliente en unidades de tiempo, las cuales pueden ser horas, minutos, segundos, esto dependerá del problema.